# Practical Introduction to CNNs 

This notebook introduces the main building blocks of a Convolutional Neural Network (CNN) using **tiny, runnable PyTorch examples**.

## Dimension reminder
For a convolution:
- `H_out = floor((H + 2P - K) / S) + 1`
- `W_out = floor((W + 2P - K) / S) + 1`

Where:
- `H, W` = input height and width
- `K` = kernel size
- `P` = padding
- `S` = stride


## 0. Setup

In [1]:
import torch # Install pytorch
# !pip install pytorch
import torch.nn as nn
torch.manual_seed(7)

## 1. A tiny CNN made of standard layers

This one example contains the usual flow:

**Conv -> ReLU -> MaxPool -> Flatten -> Dropout -> Dense -> Softmax**

It helps students see how all the common CNN layers connect in one pipeline.


In [2]:
model = nn.Sequential(nn.Conv2d(1,4,3,padding=1), 
                      nn.ReLU(), 
                      nn.MaxPool2d(2), 
                      nn.Flatten(), 
                      nn.Dropout(0.2), 
                      nn.Linear(16,3), 
                      nn.Softmax(dim=1)) # MODEL of CNN
x = torch.arange(1., 17.).reshape(1,1,4,4) # INPUT
model(x) # Model on INPUT X
# (Input_Channels, Output_Channels, Kernel_size, stride, padding)
# For color image 3, for a gray scale image - 1

tensor([[0.2056, 0.0064, 0.7880]], grad_fn=<SoftmaxBackward0>)

The output has shape **(1, 3)** because the final dense layer maps the flattened features into 3 class scores, and softmax converts them into class probabilities that sum to 1.


## 2. Convolution layer examples

### 2.1 Input 3x3, kernel 3x3, no padding, stride 1

A 3x3 kernel fully covers the 3x3 image only once, so the output becomes **1x1**.


In [22]:
x = torch.arange(1., 10.).reshape(1,1,3,3)
conv = nn.Conv2d(1,1,3,bias=False); conv.weight.data.fill_(1.0)
y = conv(x); {"shape": tuple(y.shape), "tensor": y}
x,print(conv),y

Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1), bias=False)


(tensor([[[[1., 2., 3.],
           [4., 5., 6.],
           [7., 8., 9.]]]]),
 None,
 tensor([[[[45.]]]], grad_fn=<ConvolutionBackward0>))

### 2.2 Input 5x5, kernel 3x3, padding 1, stride 1

Padding keeps the spatial size, so a **5x5** input stays **5x5** after convolution.


In [23]:
x = torch.arange(1., 26.).reshape(1,1,5,5)
conv = nn.Conv2d(1,1,3,padding=1,bias=False); conv.weight.data.fill_(1.0)
y = conv(x); {"shape": tuple(y.shape), "tensor": y}
y.shape

torch.Size([1, 1, 5, 5])

### 2.3 Input 7x7, kernel 5x5, no padding, stride 2

A larger kernel and larger stride shrink the output more aggressively.


In [24]:
x = torch.arange(1., 50.).reshape(1,1,7,7)
conv = nn.Conv2d(1,1,5,stride=2,bias=False); conv.weight.data.fill_(1.0)
y = conv(x); {"shape": tuple(y.shape), "tensor": y}
y.shape

torch.Size([1, 1, 2, 2])

In [25]:
(7-5+0)/2+1

2.0

### 2.4 Input 11x11, kernel 7x7, padding 3, stride 2

Padding adds a border, while stride 2 downsamples. This shows how **padding and stride compete**.


In [26]:
x = torch.arange(1., 122.).reshape(1,1,11,11)
conv = nn.Conv2d(1,1,7,stride=2,padding=3,bias=False); 
conv.weight.data.fill_(1.0)
y = conv(x); {"shape": tuple(y.shape), "tensor": y}
y.shape

torch.Size([1, 1, 6, 6])

### 2.5 Same input, different number of filters

The number of filters controls the **depth** (number of output channels), not the spatial height and width.


In [27]:
x = torch.arange(1., 26.).reshape(1,1,5,5)
conv = nn.Conv2d(1,3,3,padding=1,bias=False)
conv.weight.data[0].fill_(1.0)
conv.weight.data[1].fill_(0.5)
conv.weight.data[2].fill_(-1.0)
y = conv(x); {"shape": tuple(y.shape), "channels": y[0]}
y.shape

torch.Size([1, 3, 5, 5])

Because there are **3 filters**, the output shape becomes **(1, 3, H, W)**.  
- More filters = more learned feature maps  
- Spatial size is controlled by kernel, stride, and padding  
- Channel depth is controlled by the number of filters


### 2.6 Comparing kernel sizes on the same 11x11 input

Below, the kernel is increased to **5x5** with stride 1 and no padding.  
Larger kernels capture wider neighborhoods but usually reduce output size more if padding is not used.


In [8]:
x = torch.arange(1., 122.).reshape(1,1,11,11)
conv = nn.Conv2d(1,1,5,bias=False); conv.weight.data.fill_(1.0)
y = conv(x); {"shape": tuple(y.shape), "corner": y[0,0,:3,:3]}

{'shape': (1, 1, 7, 7),
 'corner': tensor([[ 625.,  650.,  675.],
         [ 900.,  925.,  950.],
         [1175., 1200., 1225.]], grad_fn=<SliceBackward0>)}

## 3. Activation functions

### 3.1 ReLU

ReLU replaces negative values with 0 and keeps positive values unchanged.


In [9]:
x = torch.tensor([[-2.0, -0.5, 0.0, 1.0, 3.0]])
act = nn.ReLU()
act(x)

tensor([[0., 0., 0., 1., 3.]])

In [ ]:
act1 = nn.Sigmoid()
act1(x)

Negative values become **0**, so ReLU creates sparsity and helps many CNNs train faster.

### 3.2 Sigmoid

Sigmoid squeezes every value into the range **(0, 1)**.


In [10]:
x = torch.tensor([[-2.0, -0.5, 0.0, 1.0, 3.0]])
act = nn.Sigmoid()
act(x)

tensor([[0.1192, 0.3775, 0.5000, 0.7311, 0.9526]])

Large negative values move near **0**, large positive values move near **1**, and 0 maps to **0.5**.

### 3.3 Tanh

Tanh squeezes values into the range **(-1, 1)** and stays centered around zero.


In [11]:
x = torch.tensor([[-2.0, -0.5, 0.0, 1.0, 3.0]])
act = nn.Tanh()
act(x)

tensor([[-0.9640, -0.4621,  0.0000,  0.7616,  0.9951]])

Tanh keeps the sign of the input and is zero-centered, unlike sigmoid.

### 3.4 Leaky ReLU

Leaky ReLU still allows a small negative slope, so negative inputs are not forced fully to zero.


In [12]:
x = torch.tensor([[-2.0, -0.5, 0.0, 1.0, 3.0]])
act = nn.LeakyReLU(0.1)
act(x)

tensor([[-0.2000, -0.0500,  0.0000,  1.0000,  3.0000]])

Negative values are reduced, not removed completely. This helps avoid 'dead' neurons.

## 4. Pooling layers

### 4.1 Max pooling, window 2x2, stride 2

Max pooling keeps the strongest value inside each window.


In [30]:
x = torch.arange(1., 17.).reshape(1,1,4,4)
pool = nn.MaxPool2d(2,2)
y = pool(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 1, 2, 2),
 'tensor': tensor([[[[ 6.,  8.],
           [14., 16.]]]])}

In [31]:
x

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

In [32]:
y

tensor([[[[ 6.,  8.],
          [14., 16.]]]])

The output is smaller because each 2x2 block is summarized by only its maximum.

### 4.2 Average pooling, window 2x2, stride 2

Average pooling keeps the mean value inside each window.


In [14]:
x = torch.arange(1., 17.).reshape(1,1,4,4)
pool = nn.AvgPool2d(2,2)
y = pool(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 1, 2, 2),
 'tensor': tensor([[[[ 3.5000,  5.5000],
           [11.5000, 13.5000]]]])}

This smooths the feature map instead of selecting only the strongest activation.

### 4.3 Max pooling, window 3x3, stride 1

A larger window overlaps more and reduces size less aggressively when stride is small.


In [15]:
x = torch.arange(1., 37.).reshape(1,1,6,6)
pool = nn.MaxPool2d(3,1)
y = pool(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 1, 4, 4),
 'tensor': tensor([[[[15., 16., 17., 18.],
           [21., 22., 23., 24.],
           [27., 28., 29., 30.],
           [33., 34., 35., 36.]]]])}

### 4.4 Average pooling, window 3x3, stride 2

Changing stride changes how quickly the spatial dimensions shrink.


In [16]:
x = torch.arange(1., 37.).reshape(1,1,6,6)
pool = nn.AvgPool2d(3,2)
y = pool(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 1, 2, 2),
 'tensor': tensor([[[[ 8., 10.],
           [20., 22.]]]])}

## 5. Flatten layer

Flatten converts multi-dimensional feature maps into a single vector so they can be passed to dense layers.


In [33]:
x = torch.arange(1., 25.).reshape(1,2,3,4)
flat = nn.Flatten()
y = flat(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 24),
 'tensor': tensor([[ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
          15., 16., 17., 18., 19., 20., 21., 22., 23., 24.]])}

In [35]:
x.shape

torch.Size([1, 2, 3, 4])

In [34]:
y.shape

torch.Size([1, 24])

## 6. Dropout

### 6.1 Dropout in training mode

Dropout randomly sets some activations to zero during training.


In [18]:
torch.manual_seed(0)
drop = nn.Dropout(p=0.5); x = torch.ones(1,8)
drop(x)

tensor([[2., 0., 0., 2., 2., 0., 0., 2.]])

Some positions become **0**, and the kept values are scaled up to preserve the expected magnitude.

### 6.2 Dropout in evaluation mode

During evaluation/inference, dropout is turned off.


In [19]:
drop.eval()
x = torch.ones(1,8)
drop(x)

tensor([[1., 1., 1., 1., 1., 1., 1., 1.]])

Now the vector passes through unchanged, because dropout is only active during training.

## 7. Dense layer

A dense (fully connected) layer mixes all input features to produce a new feature vector.


In [20]:
x = torch.arange(1., 7.).reshape(1,6)
fc = nn.Linear(6,3,bias=False); fc.weight.data.fill_(0.5)
y = fc(x); {"shape": tuple(y.shape), "tensor": y}

{'shape': (1, 3),
 'tensor': tensor([[10.5000, 10.5000, 10.5000]], grad_fn=<MmBackward0>)}

The output has shape **(1, 3)** because 6 input features are mapped to 3 learned combinations.

## 8. Softmax

Softmax converts raw scores (logits) into probabilities.


In [21]:
x = torch.tensor([[2.0, 1.0, 0.1, -1.0]])
sm = nn.Softmax(dim=1)
sm(x)

tensor([[0.6381, 0.2347, 0.0954, 0.0318]])

All values become positive probabilities between 0 and 1, and the row sums to **1.0**.

## 9. Key takeaways

- **Convolution** learns local patterns such as edges, corners, and textures  
- **Kernel size** controls how large a neighborhood the filter looks at  
- **Padding** can preserve spatial size  
- **Stride** reduces spatial size by skipping positions  
- **Number of filters** controls output depth (channels)  
- **Activation functions** make the network non-linear  
- **Pooling** reduces spatial size and summarizes features  
- **Flatten** prepares feature maps for dense layers  
- **Dropout** regularizes the model during training  
- **Dense + Softmax** convert learned features into class probabilities  

You can now ask students to modify:
1. kernel size  
2. stride  
3. padding  
4. number of filters  
5. pooling window  
and observe how the output shapes and values change.
